In [1]:
import sys
import os
import csv
from gpt_suite import gpt_mp_handler
import pandas as pd
import random
import json
from tqdm.auto import tqdm
import re
from datetime import datetime

In [2]:
criteria_dict = json.load(open('../../automated_verification/outputs/norm_relevance/final_filtered_criteria.json'))

In [3]:
quantifier_msg = """In the first step of the process, you are a helpful quantifier assistant. You quantify the output of different tasks based on the given criteria.
The criterion is given in a dictionary format where each key is a distinct criteria.
The value of each key is a dictionary as follows {"description": criteria description , "accepted_values": possible accepted inputs for this key}
You are going to quantify each of the crieria for a given task based on the task decription.
Return a dictionary where the keys are the criteria and the values are the assessed performance based on accepted values for each criteria.
Return only the dictionary as a json format string and nothing else."""

In [4]:
DEFAULT_SYSTEM_PROMPT = """
You are a helpful assistant. Your task is to judge the relevance of a Chinese cultural social norm to the sitatution in a given conversation. Consider factors such as age of the people involved, relationships between them, settings of the conversations such as work, family or friends, topic of conversation and so on. Respond with "relevant"/"irrelevant" label and provide a justifcation for your decision.
Strictly follow response format.

Response Format: 
Justification: <justification>
Relevance: <decision>
""".strip()

In [5]:
final_judgment = "Now given the above criteria and inputs provided above, provide relevance judgment and justification.. Strictly adhere to the 'Response Format' provided in the task instructions."

def symbolic_annotator(quantifier_prompts) -> list:
    config = {"temperature": 0, "max_tokens": 500}
    model = 'gpt-4o-mini'
    handler = gpt_mp_handler.GPTMPHandler(api_key=openai_key, gen_conf=config, num_worker=10)
    results = list()
    batch = []
    for norm_id in quantifier_prompts:
        ins = {
            'init_context': '',
            'questions': [quantifier_prompts[norm_id], final_judgment],
            'task_desc': DEFAULT_SYSTEM_PROMPT,
            'debug_log': debug_dir,
            'model_name': model
        }
        batch.append(ins)
        results.append([norm_id])
    handler.add_batch(batch)
    outs = handler.process()

    # print(outs)
    for idx, out in enumerate(outs):
        if len(out) == 0:
            print("ERROR:Missing...")
            results[idx].append(-1)
            continue
        for ques, resp in out.items():
            results[idx].append(resp)
    return results

In [6]:
norm_prompts = json.load(open('/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_prompts.json'))

In [7]:
relevance_output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/norm_verification_outputs/'
bnames = os.listdir(relevance_output_dir)
resps = {}
outs = {}
for bname in tqdm(bnames):
    bres = json.load(open(os.path.join(relevance_output_dir, bname)))
    for out in bres:
        n_id, ann = out
        outs[n_id] = ann.strip()
            
print(len(outs))

  0%|          | 0/64 [00:00<?, ?it/s]

63779


In [8]:
print(len(norm_prompts[1]))

63779


In [9]:
quantifier_prompts = {}

prompts = norm_prompts[1]

for n_id in tqdm(prompts):
    test_case = prompts[n_id].strip()
    message = quantifier_msg + \
    "\n\nEvaluation dictionary: " + str(criteria_dict) + \
    "\n\nActual test case to evaluate:\n" + test_case
    quantifier_prompts[int(n_id)] = message.strip()

  0%|          | 0/63779 [00:00<?, ?it/s]

In [11]:
# print(quantifier_prompts[1])

In [12]:
print(len(quantifier_prompts))

63779


In [13]:
# openai_key = '<put-your-key-here>'
openai_key = '<put-your-key-here>'

In [14]:
debug_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/relevance_agenteval_logs/'
os.makedirs(debug_dir, exist_ok=True)

In [15]:
quantifier_prompt_keys = sorted(list(quantifier_prompts.keys()))

batch_id = 0
batches = []
b = 0
bsz = 1000
e = bsz
while b < len(quantifier_prompt_keys):
    batch_keys = quantifier_prompt_keys[b:e]
    batch = {}
    for bkey in batch_keys:
        batch[bkey] = quantifier_prompts[bkey]
    batches.append(batch)
    b += bsz
    e += bsz

print(len(batches))

64


In [16]:
output_dir = '/homes/rpujari/scratch_ml/DARPA/automated_verification/relevance_agenteval_outputs/'
os.makedirs(output_dir, exist_ok=True)

t1 = datetime.now()
for i, batch in tqdm(enumerate(batches)):
    batch_out_path = os.path.join(output_dir, f"batch_{i}.json")
    if not os.path.exists(batch_out_path):
        batch_res = symbolic_annotator(batch)
        json.dump(batch_res, open(batch_out_path, 'w'))
    t2 = datetime.now()
    print(f'batch {i} done.', t2-t1)

0it [00:00, ?it/s]

batch 0 done. 0:00:00.010956
batch 1 done. 0:00:00.011222
batch 2 done. 0:00:00.011439
batch 3 done. 0:00:00.011661
batch 4 done. 0:00:00.011863
batch 5 done. 0:00:00.012055
batch 6 done. 0:00:00.012254
batch 7 done. 0:00:00.012465
batch 8 done. 0:00:00.012669
batch 9 done. 0:00:00.012885
batch 10 done. 0:00:00.013114
batch 11 done. 0:00:00.013355
batch 12 done. 0:00:00.013613
batch 13 done. 0:00:00.013874
batch 14 done. 0:00:00.014150
batch 15 done. 0:00:00.014349
batch 16 done. 0:00:00.014619
batch 17 done. 0:00:00.014883
batch 18 done. 0:00:00.015163
batch 19 done. 0:00:00.015357
batch 20 done. 0:00:00.015638
batch 21 done. 0:00:00.015884
batch 22 done. 0:00:00.016153
batch 23 done. 0:00:00.016365
batch 24 done. 0:00:00.016633
batch 25 done. 0:00:00.016871
batch 26 done. 0:00:00.017159
batch 27 done. 0:00:00.017447
batch 28 done. 0:00:00.017645
batch 29 done. 0:00:00.017890
batch 30 done. 0:00:00.018182
batch 31 done. 0:00:00.018478
batch 32 done. 0:00:00.018673
batch 33 done. 0:00:

Verifying Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/1000 [00:00<?, ?it/s]

batch 62 done. 0:30:42.307396


Verifying Batch:   0%|          | 0/779 [00:00<?, ?it/s]

Processing Batch:   0%|          | 0/779 [00:00<?, ?it/s]

batch 63 done. 0:40:57.608986
